# **Deep Learning : Dinosaur Challenge Modeling and Evaluation**

**Team Members**:

**Course**: Artificial Intelligence, Thomas More

**Introduction:**

This notebook builds, trains, and evaluates a deep learning model for classifying dinosaur images into seven species: Ankylosaurus, Diplodocus, Parasaurolophus, Stegosaurus, Tyrannosaurus Rex, Triceratops, and Velociraptor. Using TensorFlow/Keras, we implement transfer learning with MobileNetV2, train the model with data generators from the analysis notebook, evaluate performance with a confusion matrix, and generate test predictions for Kaggle submission. A GenAI section reflects on AI assistance.

**Objectives**

- Build a model using transfer learning with MobileNetV2.
- Train the model and visualize training/validation performance.
- Evaluate the model using a confusion matrix and classification metrics.
- Generate test predictions in the Kaggle format.

**Import required libraries for modeling, training, and evaluation**

In [ ]:
import tensorflow as tf  # TensorFlow for building and training the model
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # For loading data generators
from tensorflow.keras.applications import MobileNetV2  # Pre-trained MobileNetV2 model
from tensorflow.keras.models import Model  # For creating the final model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout  # Layers for custom head
from tensorflow.keras.optimizers import Adam  # Optimizer for training
import numpy as np  # For numerical operations
import pandas as pd  # For handling submission dataframe
import matplotlib.pyplot as plt  # For plotting training curves
import seaborn as sns  # For confusion matrix visualization
from sklearn.metrics import confusion_matrix, classification_report  # For model evaluation
import json  # For loading metadata

**Define constants to match analysis script**

In [ ]:
IMG_SIZE = (224, 224)  # Image size for MobileNetV2 input
BATCH_SIZE = 32  # Batch size for data generators
NUM_CLASSES = 7  # Number of dinosaur species
EPOCHS = 20  # Maximum number of training epochs

**Class labels (same as analysis script)**

In [ ]:
class_labels = {
    0: 'Ankylosaurus',
    1: 'Diplodocus',
    2: 'Parasaurolophus',
    3: 'Stegosaurus',
    4: 'Tyrannosaurus Rex',
    5: 'Triceratops',
    6: 'Velociraptor'
}

# **--- Load Prepared Data ---**

**Load metadata from analysis script**

In [ ]:
with open('data_metadata.json', 'r') as f:  # Open metadata JSON file
    metadata = json.load(f)  # Load metadata (class indices, test IDs, etc.)

**Define directories (adjust paths to match analysis script)**

In [ ]:
train_dir = 'Data/train'  # Training data directory
test_dir = 'Data/test'  # Test data directory

**Recreate training data generator with same settings as analysis script**

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Rescale pixel values to [0,1]
    rotation_range=20,  # Random rotation up to 20 degrees
    width_shift_range=0.2,  # Random horizontal shift
    height_shift_range=0.2,  # Random vertical shift
    shear_range=0.2,  # Shear transformation
    zoom_range=0.2,  # Random zoom
    horizontal_flip=True,  # Random horizontal flip
    fill_mode='nearest',  # Fill new pixels with nearest value
    validation_split=0.2  # 20% validation split
)

**Recreate test data generator**

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)  # Only rescale test images

**Load training data generator**

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,  # Training directory
    target_size=IMG_SIZE,  # Resize images to 224x224
    batch_size=BATCH_SIZE,  # 32 images per batch
    class_mode='categorical',  # One-hot encoded labels
    subset='training'  # Training subset
)

**Load validation data generator**

In [ ]:
validation_generator = train_datagen.flow_from_directory(
    train_dir,  # Training directory
    target_size=IMG_SIZE,  # Resize images
    batch_size=BATCH_SIZE,  # 32 images per batch
    class_mode='categorical',  # One-hot encoded labels
    subset='validation'  # Validation subset
)

**Load test data from metadata**

In [ ]:
test_df = pd.DataFrame({
    'id': metadata['test_ids'],  # Test image IDs
    'filename': metadata['test_filenames']  # Test image paths
})
test_generator = test_datagen.flow_from_dataframe(
    test_df,  # Dataframe with test data
    x_col='filename',  # Column with file paths
    y_col=None,  # No labels
    target_size=IMG_SIZE,  # Resize images
    batch_size=BATCH_SIZE,  # 32 images per batch
    class_mode=None,  # No labels
    shuffle=False  # Preserve order
)


**Verify class indices match metadata**

In [ ]:
print('Class indices:', train_generator.class_indices)  # Print current class indices
print('Loaded from metadata:', metadata['class_indices'])  # Print saved indices
assert train_generator.class_indices == metadata['class_indices'], 'Class indices mismatch!'  # Check for consistency

## **--- Model Development with Transfer Learning ---**

**Load pre-trained MobileNetV2 model**

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',  # Use ImageNet pre-trained weights
    include_top=False,  # Exclude top classification layer
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)  # Input shape: 224x224x3 (RGB)
)


**Freeze base model layers to preserve pre-trained weights**

base_model.trainable = False  # Prevent training of base model layers

**Add custom layers for classification**

In [ ]:
x = base_model.output  # Get output from base model
x = GlobalAveragePooling2D()(x)  # Reduce spatial dimensions to a vector
x = Dense(512, activation='relu')(x)  # Add dense layer with 512 units
x = Dropout(0.5)(x)  # Add 50% dropout to prevent overfitting
predictions = Dense(NUM_CLASSES, activation='softmax')(x)  # Output layer for 7 classes

**Create final model**

In [ ]:
model = Model(inputs=base_model.input, outputs=predictions)  # Combine base and custom layers

**Compile model**

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),  # Use Adam optimizer with 0.001 learning rate
    loss='categorical_crossentropy',  # Loss function for multi-class classification
    metrics=['accuracy']  # Track accuracy during training
)

**Print model summary**

In [ ]:
model.summary()  # Display model architecture and parameters

## **--- Model Training ---**

**Train the model**

In [ ]:
history = model.fit(
    train_generator,  # Training data
    epochs=EPOCHS,  # Maximum 20 epochs
    validation_data=validation_generator,  # Validation data
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=5,  # Stop after 5 epochs with no improvement
            restore_best_weights=True  # Restore best weights
        )
    ]
)

**Plot training and validation loss/accuracy**

In [ ]:
plt.figure(figsize=(12, 4))  # Set figure size for two subplots

**Plot loss**

In [ ]:
plt.subplot(1, 2, 1)  # First subplot
plt.plot(history.history['loss'], label='Training Loss')  # Plot training loss
plt.plot(history.history['val_loss'], label='Validation Loss')  # Plot validation loss
plt.title('Loss Over Time')  # Add title
plt.xlabel('Epoch')  # Label x-axis
plt.ylabel('Loss')  # Label y-axis
plt.legend()  # Show legend

**Plot accuracy**

In [ ]:
plt.subplot(1, 2, 2)  # Second subplot
plt.plot(history.history['accuracy'], label='Training Accuracy')  # Plot training accuracy
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')  # Plot validation accuracy
plt.title('Accuracy Over Time')  # Add title
plt.xlabel('Epoch')  # Label x-axis
plt.ylabel('Accuracy')  # Label y-axis
plt.legend()  # Show legend

plt.savefig('training_curves.png')  # Save plot
plt.close()  # Close plot


## **--- Model Evaluation ---**

**Get validation predictions**

In [ ]:
validation_generator.reset()  # Reset generator to start
y_pred = model.predict(validation_generator)  # Predict on validation data
y_pred_classes = np.argmax(y_pred, axis=1)  # Convert probabilities to class indices
y_true = validation_generator.classes  # Get true class labels

**Compute confusion matrix**

In [ ]:
cm = confusion_matrix(y_true, y_pred_classes)  # Calculate confusion matrix

**Plot confusion matrix**

In [ ]:
plt.figure(figsize=(10, 8))  # Set figure size
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',  # Plot heatmap with annotations
    xticklabels=class_labels.values(), yticklabels=class_labels.values()  # Label axes with class names
)
plt.title('Confusion Matrix')  # Add title
plt.xlabel('Predicted')  # Label x-axis
plt.ylabel('True')  # Label y-axis
plt.savefig('confusion_matrix.png')  # Save plot
plt.close()  # Close plot

**Print classification report**

In [ ]:
print(classification_report(
    y_true, y_pred_classes, target_names=class_labels.values()  # Generate report with metrics
))

## **--- Test Set Predictions ---**

**Generate test predictions**

In [ ]:
test_generator.reset()  # Reset generator
test_preds = model.predict(test_generator)  # Predict on test data
test_pred_classes = np.argmax(test_preds, axis=1)  # Convert to class indices

**Create submission dataframe**

In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],  # Test image IDs
    'label': test_pred_classes  # Predicted labels
})

**Save submission file**

In [ ]:
submission.to_csv('submission.csv', index=False)  # Save as CSV for Kaggle
print(submission.head())  # Display first few rows